In [4]:

import csv
from pathlib import Path

DATA_DIR = Path("../data")
soil_file = DATA_DIR / "soil.csv"

print("Soil dataset")
print("-" * 40)

if not soil_file.exists():
    print("Error: soil.csv was not found.")
    print("Expected location:", soil_file.resolve())
else:
    with open(soil_file, "r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        columns = reader.fieldnames
        rows = list(reader)

    print("Rows:", len(rows))
    print("Columns:", columns)
    print()

    print("First 5 records:")
    for row in rows[:5]:
        print(row)

Soil dataset
----------------------------------------
Rows: 23
Columns: ['district', 'soil_type', 'soil_texture', 'drainage', 'soil_depth', 'ph_category', 'organic_matter_category', 'fertility_category']

First 5 records:
{'district': 'Thiruvananthapuram', 'soil_type': 'Red soil', 'soil_texture': 'Red loam', 'drainage': 'Well drained', 'soil_depth': 'Deep', 'ph_category': 'Slightly acidic to neutral', 'organic_matter_category': 'Low', 'fertility_category': 'Low'}
{'district': 'Thiruvananthapuram', 'soil_type': 'Gravelly laterite', 'soil_texture': 'Gravelly', 'drainage': 'Well drained', 'soil_depth': 'Deep', 'ph_category': 'Strongly acidic', 'organic_matter_category': 'Low', 'fertility_category': 'Low'}
{'district': 'Kollam', 'soil_type': 'Onattukara sandy soil', 'soil_texture': 'Sandy', 'drainage': 'Well drained', 'soil_depth': 'Very deep', 'ph_category': 'Strongly acidic', 'organic_matter_category': 'Moderate', 'fertility_category': 'Low'}
{'district': 'Kollam', 'soil_type': 'Coastal 

In [6]:

import csv
from pathlib import Path

soil_file = Path("../data/soil.csv")

with open(soil_file, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    rows = list(reader)

print("GROW - SOIL DATASET SUMMARY")
print("=" * 40)

print("Total records:", len(rows))
print("Unique districts:", len(set(row["district"] for row in rows)))
print("Unique soil types:", len(set(row["soil_type"] for row in rows)))

print("\nDistricts included:")
for district in sorted(set(row["district"] for row in rows)):
    print("-", district)

print("\nSoil types included:")
for soil_type in sorted(set(row["soil_type"] for row in rows)):
    print("-", soil_type)


GROW - SOIL DATASET SUMMARY
Total records: 23
Unique districts: 13
Unique soil types: 15

Districts included:
- Alappuzha
- Ernakulam
- Idukki
- Kannur
- Kasaragod
- Kollam
- Kottayam
- Kozhikode
- Malappuram
- Palakkad
- Thiruvananthapuram
- Thrissur
- Wayanad

Soil types included:
- Black soil
- Brown hydromorphic soil
- Coastal sandy soil
- Forest soil
- Gravelly laterite
- Kaipad soil
- Kole soil
- Kuttanad soil
- Marayoor soil
- Onattukara sandy soil
- Pokkali soil
- Poonthalpadam soil
- Red loam soil
- Red soil
- Riverine alluvium


In [8]:
import csv
from pathlib import Path

crop_file = Path("../data/crops.csv")

with open(crop_file, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)
    crops = list(reader)

print("GROW - CROP DATASET")
print("-" * 40)
print("Total crops:", len(crops))
print("Columns:", list(crops[0].keys()))

print("\nCrop names:")
for crop in crops:
    print("-", crop["crop"])

GROW - CROP DATASET
----------------------------------------
Total crops: 10
Columns: ['crop', 'preferred_soil_texture', 'preferred_drainage', 'preferred_ph_range', 'water_requirement', 'growing_season']

Crop names:
- Rice
- Coconut
- Banana
- Black pepper
- Rubber
- Tapioca
- Ginger
- Turmeric
- Groundnut
- Vegetables


In [ ]:
import csv
from pathlib import Path

DATA_DIR = Path("../data")

soil_file = DATA_DIR / "soil.csv"
crop_file = DATA_DIR / "crops.csv"
output_file = DATA_DIR / "soil_crop_suitability.csv"


# Read soil dataset
with open(soil_file, "r", newline="", encoding="utf-8-sig") as file:
    soil_records = list(csv.DictReader(file))

# Read crop dataset
with open(crop_file, "r", newline="", encoding="utf-8-sig") as file:
    crop_records = list(csv.DictReader(file))


def score_texture(soil_texture, preferred_texture):
    """Preliminary heuristic: compare soil texture descriptions."""
    soil_text = soil_texture.lower()
    preferred = preferred_texture.lower()

    if soil_text == preferred:
        return 100

    if preferred in soil_text or soil_text in preferred:
        return 80

    # Broad texture-group matches; these are prototype rules.
    if "loam" in soil_text and "loam" in preferred:
        return 70

    if "clay" in soil_text and "clay" in preferred:
        return 70

    if "sand" in soil_text and "sand" in preferred:
        return 70

    return 40


def score_drainage(soil_drainage, preferred_drainage):
    """Preliminary drainage compatibility score."""
    actual = soil_drainage.lower()
    preferred = preferred_drainage.lower()

    if actual == preferred:
        return 100

    if "poorly drained" in actual and "well drained" in preferred:
        return 20

    if "well drained" in actual and "poorly drained" in preferred:
        return 20

    return 50


results = []

for soil in soil_records:
    for crop in crop_records:
        texture_score = score_texture(
            soil["soil_texture"],
            crop["preferred_soil_texture"]
        )

        drainage_score = score_drainage(
            soil["drainage"],
            crop["preferred_drainage"]
        )

        # Equal weighting for the initial prototype.
        suitability_score = round(
            (texture_score + drainage_score) / 2,
            1
        )

        results.append({
            "district": soil["district"],
            "soil_type": soil["soil_type"],
            "crop": crop["crop"],
            "texture_score": texture_score,
            "drainage_score": drainage_score,
            "suitability_score": suitability_score
        })


# Save the results
with open(output_file, "w", newline="", encoding="utf-8") as file:
    fieldnames = [
        "district",
        "soil_type",
        "crop",
        "texture_score",
        "drainage_score",
        "suitability_score"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


print("GROW - SOIL-CROP SUITABILITY ANALYSIS")
print("-" * 45)
print("Soil records:", len(soil_records))
print("Crop records:", len(crop_records))
print("Suitability records:", len(results))
print("Output file:", output_file.resolve())

print("\nFirst 10 results:")

for result in results[:10]:
    print(
        result["district"],
        "|", result["crop"],
        "| Score:", result["suitability_score"]
    )

In [9]:
import csv
from pathlib import Path

DATA_DIR = Path("../data")

soil_file = DATA_DIR / "soil.csv"
crop_file = DATA_DIR / "crops.csv"
output_file = DATA_DIR / "soil_crop_suitability.csv"


# Read soil dataset
with open(soil_file, "r", newline="", encoding="utf-8-sig") as file:
    soil_records = list(csv.DictReader(file))

# Read crop dataset
with open(crop_file, "r", newline="", encoding="utf-8-sig") as file:
    crop_records = list(csv.DictReader(file))


def score_texture(soil_texture, preferred_texture):
    """Preliminary heuristic: compare soil texture descriptions."""
    soil_text = soil_texture.lower()
    preferred = preferred_texture.lower()

    if soil_text == preferred:
        return 100

    if preferred in soil_text or soil_text in preferred:
        return 80

    # Broad texture-group matches; these are prototype rules.
    if "loam" in soil_text and "loam" in preferred:
        return 70

    if "clay" in soil_text and "clay" in preferred:
        return 70

    if "sand" in soil_text and "sand" in preferred:
        return 70

    return 40


def score_drainage(soil_drainage, preferred_drainage):
    """Preliminary drainage compatibility score."""
    actual = soil_drainage.lower()
    preferred = preferred_drainage.lower()

    if actual == preferred:
        return 100

    if "poorly drained" in actual and "well drained" in preferred:
        return 20

    if "well drained" in actual and "poorly drained" in preferred:
        return 20

    return 50


results = []

for soil in soil_records:
    for crop in crop_records:
        texture_score = score_texture(
            soil["soil_texture"],
            crop["preferred_soil_texture"]
        )

        drainage_score = score_drainage(
            soil["drainage"],
            crop["preferred_drainage"]
        )

        # Equal weighting for the initial prototype.
        suitability_score = round(
            (texture_score + drainage_score) / 2,
            1
        )

        results.append({
            "district": soil["district"],
            "soil_type": soil["soil_type"],
            "crop": crop["crop"],
            "texture_score": texture_score,
            "drainage_score": drainage_score,
            "suitability_score": suitability_score
        })


# Save the results
with open(output_file, "w", newline="", encoding="utf-8") as file:
    fieldnames = [
        "district",
        "soil_type",
        "crop",
        "texture_score",
        "drainage_score",
        "suitability_score"
    ]

    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


print("GROW - SOIL-CROP SUITABILITY ANALYSIS")
print("-" * 45)
print("Soil records:", len(soil_records))
print("Crop records:", len(crop_records))
print("Suitability records:", len(results))
print("Output file:", output_file.resolve())

print("\nFirst 10 results:")

for result in results[:10]:
    print(
        result["district"],
        "|", result["crop"],
        "| Score:", result["suitability_score"]
    )

GROW - SOIL-CROP SUITABILITY ANALYSIS
---------------------------------------------
Soil records: 23
Crop records: 10
Suitability records: 230
Output file: C:\Users\Sidhu\Desktop\new\GROW\data\soil_crop_suitability.csv

First 10 results:
Thiruvananthapuram | Rice | Score: 30.0
Thiruvananthapuram | Coconut | Score: 85.0
Thiruvananthapuram | Banana | Score: 90.0
Thiruvananthapuram | Black pepper | Score: 90.0
Thiruvananthapuram | Rubber | Score: 90.0
Thiruvananthapuram | Tapioca | Score: 85.0
Thiruvananthapuram | Ginger | Score: 90.0
Thiruvananthapuram | Turmeric | Score: 90.0
Thiruvananthapuram | Groundnut | Score: 85.0
Thiruvananthapuram | Vegetables | Score: 90.0


In [7]:
import csv
from pathlib import Path

soil_file = Path("../data/soil.csv")

with open(soil_file, "r", newline="", encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))

print("MISSING VALUE REPORT")
print("=" * 40)

if not rows:
    print("The dataset has no records.")
else:
    columns = rows[0].keys()

    for column in columns:
        missing = sum(
            1 for row in rows
            if row[column] is None or not row[column].strip()
        )
        print(f"{column}: {missing} missing values")


MISSING VALUE REPORT
district: 0 missing values
soil_type: 0 missing values
soil_texture: 0 missing values
drainage: 0 missing values
soil_depth: 0 missing values
ph_category: 0 missing values
organic_matter_category: 0 missing values
fertility_category: 0 missing values
